In [1]:
import sys
sys.path.append("../src")

In [2]:
from domain import (
    Base,
    engine,
    Session,
    Employee,
    AppCredential,
    Category,
    Allergen,
    Dish,
    Menu,
    Sale
)
from datetime import time

In [3]:
### Categories i Al·lèrgens ###

with Session() as session:
    # 1. CATEGORIES
    category_names = ["Entrants", "Plats Principals", "Postres", "Begudes", "Tapes", "Salses"]

    existing_categories = {c.category_name: c for c in session.query(Category).all()}
    inserted_cat = 0
    for name in category_names:
        if name not in existing_categories:
            category = Category(category_name=name)
            session.add(category)
            session.flush() # Ens assegurem de tenir l'ID
            existing_categories[name] = category
            inserted_cat += 1

    # 2. ALLERGENS
    allergen_names = ["Gluten", "Lactis", "Fruits secs", "Crustacis", "Ou", "Soja", "Peix", "Sèsam"]

    existing_allergens = {a.allergen_name: a for a in session.query(Allergen).all()}
    inserted_all = 0
    for name in allergen_names:
        if name not in existing_allergens:
            allergen = Allergen(allergen_name=name)
            session.add(allergen)
            session.flush()
            existing_allergens[name] = allergen
            inserted_all += 1

    session.commit()
    print(f"Inserides {inserted_cat} categories i {inserted_all} al·lèrgens.")

Inserides 6 categories i 8 al·lèrgens.


In [4]:
### Empleats i Credencials ###
#Crea els empleats i aprofita la relació 1:1 per generar credencials de l'aplicació.

with Session() as session:
    # 3. EMPLOYEES & CREDENTIALS
    employees_data = [
        ("Joan", "Pérez", "11111111A", "Cambrer", "joan@restaurant.cat", "hash_joan_123"),
        ("Maria", "García", "22222222B", "Cuinera", "maria@restaurant.cat", "hash_maria_456"),
        ("Marc", "Vila", "33333333C", "Encarregat", "marc@restaurant.cat", "hash_marc_789"),
        ("Anna", "Serra", "44444444D", "Cambrera", "anna@restaurant.cat", "hash_anna_321"),
        ("Lluís", "Font", "55555555E", "Cuiner", "lluis@restaurant.cat", "hash_lluis_654"),
        ("Marta", "Roca", "66666666F", "Cambrera", "marta@restaurant.cat", "hash_marta_987"),
    ]

    existing_employees = {e.national_id for (e,) in session.query(Employee.national_id).all()}
    inserted_emp = 0

    for name, surname, national_id, role, email, password_hash in employees_data:
        if national_id not in existing_employees:
            employee = Employee(
                name=name,
                surname=surname,
                national_id=national_id,
                role=role
            )

            # Creació lligada a la relació 1:1
            employee.credentials = AppCredential(
                email=email,
                password_hash=password_hash
            )
            session.add(employee)
            inserted_emp += 1

    session.commit()
    print(f"Inserits {inserted_emp} empleats amb les seves credencials.")

Inserits 6 empleats amb les seves credencials.


In [5]:
### Plats (Dishes) ###
#Aquí relacionem ategories creades a la Cel·la 1 amb la taula N:M dels al·lèrgens.

with Session() as session:
    # 4. DISHES
    categories = {c.category_name: c for c in session.query(Category).all()}
    allergens = {a.allergen_name: a for a in session.query(Allergen).all()}

    dishes_data = [
        {
            "name": "Amanida Cèsar",
            "desc": "Amanida fresca amb pollastre i crostons",
            "category": "Entrants",
            "allergens": ["Gluten", "Lactis", "Ou"]
        },
        {
            "name": "Macarrons a la bolonyesa",
            "desc": "Pasta amb salsa de tomàquet i carn picada",
            "category": "Plats Principals",
            "allergens": ["Gluten"]
        },
        {
            "name": "Lluç al forn",
            "desc": "Lluç fresc amb patates fornades",
            "category": "Plats Principals",
            "allergens": []
        },
        {
            "name": "Pastís de formatge",
            "desc": "Cheesecake casolà",
            "category": "Postres",
            "allergens": ["Lactis", "Ou", "Gluten"]
        },
        {
            "name": "Aigua mineral",
            "desc": "Ampolla de 500ml",
            "category": "Begudes",
            "allergens": []
        },
        {
        "name": "Patates Braves",
        "desc": "Patates cruixents amb salsa brava casolana",
        "category": "Tapes",
        "allergens": []
        },
        {
            "name": "Calamars a la romana",
            "desc": "Anelles de calamar arrebossades",
            "category": "Tapes",
            "allergens": ["Gluten", "Peix", "Ou"]
        },
        {
            "name": "Entrecot a la brasa",
            "desc": "Entrecot de vedella de 300g amb guarnició",
            "category": "Plats Principals",
            "allergens": []
        },
        {
            "name": "Coulant de xocolata",
            "desc": "Coulant calent amb interior líquid",
            "category": "Postres",
            "allergens": ["Gluten", "Lactis", "Ou"]
        },
        {
            "name": "Vi negre DO Penedès",
            "desc": "Copa de vi negre",
            "category": "Begudes",
            "allergens": []
        }
    ]

    existing_dishes = {d.dish_name: d for d in session.query(Dish).all()}
    inserted_dishes = 0

    for item in dishes_data:
        if item["name"] not in existing_dishes:
            dish = Dish(
                dish_name=item["name"],
                description=item["desc"],
                is_active=True,
                id_category=categories[item["category"]].id
            )
            # Associem al·lèrgens (Relació Many-to-Many)
            dish.allergens = [allergens[a] for a in item["allergens"]]

            session.add(dish)
            inserted_dishes += 1

    session.commit()
    print(f"Inserits {inserted_dishes} plats amb els seus al·lèrgens.")

Inserits 10 plats amb els seus al·lèrgens.


In [6]:
### Menús ###
#Agrupem els plats existents per formar diferents menús.
with Session() as session:
    # 5. MENUS
    dishes = {d.dish_name: d for d in session.query(Dish).all()}

    menus_data = [
        ("Menú Diari", 12.50, ["Amanida Cèsar", "Macarrons a la bolonyesa", "Aigua mineral"]),
        ("Menú Nit", 18.00, ["Amanida Cèsar", "Lluç al forn", "Pastís de formatge", "Aigua mineral"]),
        ("Menú Cap de Setmana", 25.00, ["Patates Braves", "Entrecot a la brasa", "Coulant de xocolata", "Vi negre DO Penedès"]),
        ("Menú Tapes", 15.00, ["Patates Braves", "Calamars a la romana", "Pastís de formatge", "Aigua mineral"]),
    ]

    existing_menus = {m.menu_name: m for m in session.query(Menu).all()}
    inserted_menus = 0

    for menu_name, price, menu_dishes in menus_data:
        if menu_name not in existing_menus:
            menu = Menu(
                menu_name=menu_name,
                price=price
            )
            # Associem plats al menú (Relació Many-to-Many)
            menu.dishes = [dishes[d_name] for d_name in menu_dishes]

            session.add(menu)
            inserted_menus += 1

    session.commit()
    print(f"Inserits {inserted_menus} menús.")

Inserits 4 menús.


In [7]:
### Vendes (Sales) ###
#L'últim bloc simula vendes reals per part dels empleats, afegint la data i la quantitat venuda.

from datetime import date

with Session() as session:
    # 6. SALES
    menus = {m.menu_name: m for m in session.query(Menu).all()}
    employees = {e.national_id: e for e in session.query(Employee).all()}

    # Data: (Nom Menú, DNI Empleat, Data, Quantitat)
    sales_data = [
        ("Menú Diari", "11111111A", date(2026, 3, 26), 2),
        ("Menú Nit", "11111111A", date(2026, 3, 26), 1),
        ("Menú Diari", "33333333C", date(2026, 3, 25), 4),
        ("Menú Cap de Setmana", "44444444D", date(2026, 3, 28), 2),
        ("Menú Tapes", "66666666F", date(2026, 3, 28), 3),
        ("Menú Nit", "11111111A", date(2026, 3, 29), 2),
        ("Menú Cap de Setmana", "33333333C", date(2026, 3, 29), 5),
    ]

    existing_sales = {
        (s.id_menu, s.id_employee, s.sale_date)
        for s in session.query(Sale).all()
    }
    inserted_sales = 0

    for menu_name, emp_dni, sale_date, quantity in sales_data:
        menu = menus.get(menu_name)
        employee = employees.get(emp_dni)

        if not menu or not employee:
            print(f"Avís: Menú '{menu_name}' o Empleat '{emp_dni}' no trobat.")
            continue

        key = (menu.id, employee.id, sale_date)
        if key in existing_sales:
            continue

        sale = Sale(
            id_menu=menu.id,
            id_employee=employee.id,
            sale_date=sale_date,
            quantity_sold=quantity
        )
        session.add(sale)
        inserted_sales += 1

    session.commit()
    print(f"Inserides {inserted_sales} vendes de menús.")

Inserides 7 vendes de menús.
